In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_hta = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_hta/")
silver_diag = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_diag")
silver_insu = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_insumos")
silver_medica = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_medicamentos")
silver_proced = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_procedimientos")
silver_nam_diag = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/diagnóstico/")
silver_dim_insum = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/insumos/")
silver_dim_med = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/medicamentos/")
silver_dim_proc = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/procedimientos/")

In [0]:
display(silver_hta)
display(silver_diag)
display(silver_insu)
display(silver_medica)
display(silver_proced)
display(silver_dim_insum)
display(silver_dim_med)
display(silver_dim_proc)
display(silver_nam_diag)

## Transformación para silver_hta (nuestro fact table)

In [0]:
silver_hta = silver_hta.withColumn(
    "SEXO",
    when(col("SEXO").contains("FEMENINO"), "F").otherwise("M")
)
silver_hta = silver_hta.drop("FECHA_CORTE")
silver_hta = silver_hta.filter(col("DEPARTAMENTO") == "LIMA")

In [0]:
silver_hta.filter(lower(col("PROVINCIA")) == "san juan de miraflores")
silver_hta.drop("DEPARTAMENTO","PROVINCIA","UBIGEO")

In [0]:
silver_hta.withColumn("Fecha_de_atencion",to_date(silver_hta.FECHA_ATENCION,"yyyyMMdd"))
silver_hta.drop("FECHA_ATENCION")

## Transformación para silver_diag

In [0]:
silver_diag = silver_diag.withColumn(
    "TIPO_DIAGNOSTICO",
    when(col("TIPO_DIAGNOSTICO").contains("DEFINITIVO"), "D")
    .otherwise(
        when(col("TIPO_DIAGNOSTICO").contains("PRESUNTIVO"), "P")
        .otherwise("R")
    )
)

In [0]:
silver_diag = silver_diag.drop(col("FECHA_CORTE"))

In [0]:
tables = {
    "hta_sis.silver_hta": silver_hta,
    "hta_sis.silver_diag": silver_diag,
    "hta_sis.silver_insu": silver_insu,
    "hta_sis.silver_medica": silver_medica,
    "hta_sis.silver_proced": silver_proced,
    "hta_sis.silver_nam_diag": silver_nam_diag,
    "hta_sis.silver_dim_insum": silver_dim_insum,
    "hta_sis.silver_dim_med": silver_dim_med,
    "hta_sis.silver_dim_proc": silver_dim_proc
}

for table_name, df in tables.items():
    if spark.catalog.tableExists(table_name):
        print(f"Tabla {table_name} existe. Haciendo overwrite...")
    else:
        print(f"Creando nueva tabla {table_name}...")
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)